<a href="https://colab.research.google.com/github/Khanbir/Data_analitycs/blob/main/DZ_14_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [ ]:
import numpy as np
from statsmodels.stats.power import NormalIndPower

# Задані параметри
p1 = 0.19
p2 = 0.20
alpha = 0.05
power = 0.80

# Розрахунок розміру ефекту
effect_size = np.sqrt((p2 - p1)**2 / (p1*(1-p1) + p2*(1-p2)))

# Розрахунок необхідного розміру вибірки на групу
power_analysis = NormalIndPower()
sample_size_per_group = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1
)

total_sample_size = np.ceil(sample_size_per_group) * 2

print(f"Необхідний розмір вибірки на групу: {np.ceil(sample_size_per_group):.0f} користувачів")
print(f"Сумарна необхідна кількість користувачів: {total_sample_size:.0f} користувачів")

Необхідний розмір вибірки на групу: 49276 користувачів
Сумарна необхідна кількість користувачів: 98552 користувачів


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Завантаження даних
df = pd.read_csv('/content/drive/MyDrive/cookie_cats.csv')

# Розрахунок середнього retention_7 для кожної групи
retention_7_mean = df.groupby('version')['retention_7'].mean()
print("Середнє значення retention_7 по версіях гри:")
print(retention_7_mean)

# Гіпотеза:
# H0: Середнє утримання (retention) однакове в обох групах.
# H1: Середнє утримання в групі 'gate_30' вище, ніж у 'gate_40'.

Середнє значення retention_7 по версіях гри:
version
gate_30    0.190201
gate_40    0.182000
Name: retention_7, dtype: float64


**Гіпотеза**:
На основі цих даних, гіпотеза полягає в тому, що контрольна група (gate_30) має краще утримання через 7 днів після встановлення гри, ніж тестова група (gate_40).

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [ ]:
# Підготовка даних для тесту
successes = df.groupby('version')['retention_7'].sum()
nobs = df.groupby('version')['retention_7'].count()

# Виконання Z-тесту для двох пропорцій
z_statistic, p_value = proportions_ztest(successes, nobs)

# Обчислення 95% довірчих інтервалів
conf_int_control = proportion_confint(successes['gate_30'], nobs['gate_30'], alpha=0.05)
conf_int_treatment = proportion_confint(successes['gate_40'], nobs['gate_40'], alpha=0.05)

print("\n--- Результати Z-тесту та довірчих інтервалів ---")
print(f"z statistic: {z_statistic:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{conf_int_control[0]:.4f}, {conf_int_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{conf_int_treatment[0]:.4f}, {conf_int_treatment[1]:.4f}]")


--- Результати Z-тесту та довірчих інтервалів ---
z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


**Висновок:**

Статистична значущість: Оскільки p-value (0.0016) менше рівня значущості
alpha (0.05), ми можемо стверджувати, що різниця між поведінкою користувачів у різних версіях гри є статистично значущою.

Довірчі інтервали: Довірчі інтервали для контрольної групи [0.1865, 0.1940] і для тестової групи [0.1783, 0.1858] не перетинаються. Це вказує на те, що різниця між пропорціями є значущою і що істинна частка утримання для gate_30 з високою ймовірністю вища за таку для gate_40.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


**Формулювання гіпотез:**

Нульова гіпотеза (H0): Немає залежності між версією гри (gate_30 або gate_40) та утриманням гравця на 7-й день. Події є незалежними.

Альтернативна гіпотеза (Ha): Існує залежність між версією гри та утриманням гравця. Події є залежними.

In [ ]:
# Створення таблиці спряженості
contingency_table = pd.crosstab(df['version'], df['retention_7'])

# Виконання тесту Хі-квадрат
chi2, p_value_chi2, dof, expected = chi2_contingency(contingency_table)

print("\n--- Результати тесту Хі-квадрат ---")
print(f"p-value: {p_value_chi2:.4f}")


--- Результати тесту Хі-квадрат ---
p-value: 0.0016


Висновок:
Оскільки p-value (0.0016) менше рівня значущості
alpha (0.05), ми відхиляємо нульову гіпотезу. Це означає, що існує залежність між версією гри та утриманням гравців.